<a href="https://colab.research.google.com/github/5ahar-K/CodeGraph-agent/blob/main/Code%20dependencies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install networkx anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.8 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/pallets/click.git target_repo

Cloning into 'target_repo'...
remote: Enumerating objects: 15406, done.
remote: Counting objects: 100% (497/497), done.
remote: Compressing objects: 100% (237/237), done.
remote: Total 15406 (delta 425), reused 260 (delta 260), pack-reused 14909 (from 3)
Receiving objects: 100% (15406/15406), 5.40 MiB | 14.01 MiB/s, done.
Resolving deltas: 100% (10518/10518), done.


In [3]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
client = genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [11]:
import ast
import os
import networkx as nx
from collections import defaultdict

def find_function_calls(func_node):
    calls = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                calls.append(node.func.id)
            elif isinstance(node.func, ast.Attribute):
                calls.append(node.func.attr)
    return calls

def get_imports(tree):
    imports = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            module = node.module or ""     #from . import echo    (importing from the current package)
            for alias in node.names:     #e.g. from click.utils import echo as e
                local_name = alias.asname or alias.name
                imports[local_name] = module
        elif isinstance(node, ast.Import):
            for alias in node.names:
                local_name = alias.asname or alias.name    #e.g import click (no from package)
                imports[local_name] = alias.name
    return imports

def module_matches_file(module, filepath):
    if not module:
        return False
    module_as_path = module.replace(".", os.sep)
    normalized_filepath = filepath.replace("\\", os.sep)
    return module_as_path in normalized_filepath

def build_graph(repo_path):
    graph = nx.DiGraph()
    name_to_keys = defaultdict(list)
    function_bodies = {}       # (file, name) -> AST node
    file_imports = {}          # file -> {imported_name: module_string}

    # Pass 1: Find all the functions
    for root, _, files in os.walk(repo_path):
        for filename in files:
            if not filename.endswith(".py"):
                continue
            filepath = os.path.join(root, filename)
            try:
                with open(filepath, encoding="utf-8") as f:
                    tree = ast.parse(f.read(), filename=filepath)
            except SyntaxError:
                continue

            file_imports[filepath] = get_imports(tree)

            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    key = (filepath, node.name)
                    graph.add_node(key, name=node.name, file=filepath, line=node.lineno)
                    name_to_keys[node.name].append(key)
                    function_bodies[key] = node

    # Pass 2: map each call to the most likely specific function
    for key, func_node in function_bodies.items():
        source_file, _ = key
        imports_in_this_file = file_imports.get(source_file, {})

        for called_name in find_function_calls(func_node):
            candidates = name_to_keys.get(called_name)
            if not candidates:
                continue  # built-in or external call, skip

            resolved = None

            # 1. Same-file definition wins first, matching Python's own scoping rules
            same_file_key = (source_file, called_name)
            if same_file_key in function_bodies:
                resolved = [same_file_key]

            # 2. Explicitly imported
            elif called_name in imports_in_this_file:
                module = imports_in_this_file[called_name]
                matched = [c for c in candidates if module_matches_file(module, c[0])]
                if matched:
                    resolved = matched

            # 3. Only one candidate anywhere
            if resolved is None and len(candidates) == 1:
                resolved = candidates

            # 4. Still ambiguous — fall back to connecting to all
            if resolved is None:
                resolved = candidates

            for target_key in resolved:
                graph.add_edge(key, target_key)

    return graph, function_bodies, name_to_keys
g, function_bodies, name_to_keys = build_graph("target_repo")
ambiguous_count = 0
for key, func_node in function_bodies.items():
    calls = find_function_calls(func_node)
    for name in calls:
        if len(name_to_keys.get(name, [])) > 1:
           ambiguous_count += 1

print(f"Calls to ambiguously-named functions: {ambiguous_count}")

Calls to ambiguously-named functions: 2093


In [10]:
ambiguous_calls = 0
resolved_to_one = 0
still_ambiguous = 0

for key, func_node in function_bodies.items():
    for called_name in find_function_calls(func_node):
        candidates = name_to_keys.get(called_name)
        if not candidates or len(candidates) <= 1:
            continue  # not ambiguous to begin with, skip entirely

        ambiguous_calls += 1

        # How many of those candidates actually got an edge in the real graph?
        actual_targets = [c for c in candidates if g.has_edge(key, c)]

        if len(actual_targets) == 1:
            resolved_to_one += 1
        elif len(actual_targets) > 1:
            still_ambiguous += 1

print(f"Calls where multiple same-named functions existed: {ambiguous_calls}")
print(f"  Narrowed down to exactly one target: {resolved_to_one}")
print(f"  Still connected to multiple targets: {still_ambiguous}")

Calls where multiple same-named functions existed: 2093
  Narrowed down to exactly one target: 251
  Still connected to multiple targets: 1842


In [12]:
def what_calls(graph, filepath, function_name):
    #Who calls this function?
    key = (filepath, function_name)
    return list(graph.predecessors(key))

def what_does_it_call(graph, filepath, function_name):
    #What does this function call?
    key = (filepath, function_name)
    return list(graph.successors(key))

In [13]:
def find_matches(graph, function_name):
    #Return all (file, name) keys in the graph matching this function name.
    return [key for key in graph.nodes if key[1] == function_name]

In [15]:
graph, _, _ = build_graph("target_repo")
matches = find_matches(graph, "echo")
for m in matches:
    print(m)

('target_repo/tests/test_utils/test_prompt.py', 'echo')
('target_repo/src/click/utils.py', 'echo')


In [16]:
def get_function_source(graph, key):
    filepath, function_name = key
    with open(filepath, encoding="utf-8") as f:
        source_text = f.read()
    tree = ast.parse(source_text)
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            return ast.get_source_segment(source_text, node)
    return None

def ask_about_function(graph, key, question):
    source = get_function_source(graph, key)
    callers = what_calls(graph, key[0], key[1])
    callees = what_does_it_call(graph, key[0], key[1])

    prompt = f"""Here is a function called `{key[1]}` from file `{key[0]}`:

{source}

It is called by: {callers}
It calls: {callees}

Question: {question}

Answer using only the information given above. If you can't determine the answer from this, say so."""

    model = client.GenerativeModel(model_name="gemini-flash-latest")
    response = model.generate_content(contents=prompt)
    return response.text

def generate_test(graph, key):
    source = get_function_source(graph, key)
    prompt = f"""Write a pytest unit test for this function:

{source}

Only output the test code, no explanation."""
    model = client.GenerativeModel(model_name="gemini-flash-latest")
    response = model.generate_content(contents=prompt)
    return response.text

In [19]:
while True:
    name = input("\nEnter a function name (or 'quit'): ")
    if name == "quit":
        break
    matches = find_matches(graph, name)
    if not matches:
        print("Function not found in graph.")
        continue
    if len(matches) > 1:
        print(f"Multiple functions named '{name}' found:")
        for i, m in enumerate(matches):
            print(f"  [{i}] {m[0]}")
        idx = int(input("Which one? Enter the number: "))
        key = matches[idx]
    else:
        key = matches[0]

    q = input("Your question (or type 'test' to generate a unit test): ")
    if q == "test":
        print(generate_test(g, key))
    else:
        print(ask_about_function(g, key, q))


Enter a function name (or 'quit'): echo
Multiple functions named 'echo' found:
  [0] target_repo/tests/test_utils/test_prompt.py
  [1] target_repo/src/click/utils.py
Which one? Enter the number: 0
Your question (or type 'test' to generate a unit test): test
```python
import click
from click.testing import CliRunner
from target_module import echo


def test_echo():
    @click.command()
    def cli():
        echo()

    runner = CliRunner()
    result = runner.invoke(cli, input="1\n2\n3\n")

    assert result.exit_code == 0
    assert result.output == "1\n2\n3\n"
```

Enter a function name (or 'quit'): quit
